# partial-CROWN versus intervalNets on a replacement Burgers PINN

This notebook compares certified pointwise Burgers residual bounds on **the same trained network**. The published ICML 2024 values are shown only as historical references because the authors did not release the Table 1 checkpoint or split history. Sampling is a diagnostic lower bound on the worst-case residual, never a certificate.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

OUTPUT = next(
    candidate
    for candidate in (
        Path('benchmark_outputs/partial_crown_burgers'),
        Path('notebooks/benchmark_outputs/partial_crown_burgers'),
    )
    if (candidate / 'comparison.json').exists()
)
payload = json.loads((OUTPUT / 'comparison.json').read_text())
rows = pd.DataFrame(payload['records'])
display(pd.Series(payload['method_status'], name='status').to_frame())
rows

## Matched same-network comparison

Tightness is measured by the certified absolute-residual upper bound divided by the sampled absolute maximum. Lower is better; the ratio is at least one when the diagnostic sample is contained. Runtime is verifier-only CPU wall time.

In [ ]:
final = (rows.sort_values('cell_evaluations').groupby('method', as_index=False).tail(1))
final[['method', 'cell_evaluations', 'residual_squared_upper', 'sampled_abs_max', 'upper_to_sample_ratio', 'verifier_time_s', 'cells_per_second']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for method, group in rows.groupby('method'):
    group = group.sort_values('cell_evaluations')
    axes[0].plot(group.cell_evaluations, group.residual_squared_upper, marker='o', label=method)
    axes[1].plot(group.verifier_time_s, group.residual_squared_upper, marker='o', label=method)
sample_squared = payload['sampled_residual_abs_max'] ** 2
for ax in axes:
    ax.axhline(sample_squared, color='black', linestyle='--', label='sampled max squared')
    ax.set_yscale('log')
    ax.set_ylabel('upper bound on |f_theta|^2')
    ax.grid(True, alpha=.25)
axes[0].set_xscale('log'); axes[0].set_xlabel('cell evaluations')
axes[1].set_xscale('log'); axes[1].set_xlabel('verifier-only wall time [s]')
axes[0].legend()
fig.tight_layout()
fig.savefig(OUTPUT / 'tightness_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

## Published reference (different unavailable checkpoint)

In [ ]:
pd.Series(payload['paper_reference'], name='published value').to_frame()